# 47 — Agentic Data Operations

### Why this exists
An agent operating data systems needs stronger boundaries than a chat assistant. This lab enforces tool authority, argument validation, freshness and an action budget.

### Learning objectives
model tools; enforce scope; detect stale state; bound actions

### Prerequisite
Python dictionaries and state

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/47-agentic-data-operations.svg)

> **Question to carry through the lab:** Where can an agent cause harm even when the model output looks reasonable?

## Capability contract

**Capability:** C09 — AI Data Systems  
**Workstream:** Agentic Data Platform  
**Primary roles:** AI Engineer|Data Engineer|Platform Engineer  
**You will prove:** Turn AI output into a verifiable data-system decision with evidence


## Agent Workbench

This capability has a practical agent-native companion in [`agent_lab/`](../agent_lab/README.md).

Before delegating work, write the **Agent Work Contract**: goal, context, constraints, tools, permissions, success criteria, failure conditions, evidence required, and escalation policy. Then inspect the proposal, design an independent verification, challenge one assumption, and record the human decision.

**Permission rule:** a model request is never an authorization decision. Tool permissions and human approval must remain enforceable outside the model.


## Mental model
Treat the model as a planner, not an authority. Execution boundaries enforce identity, permissions, argument constraints, idempotency and action budgets.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

In [ ]:
state={"table_version":7,"rows":1000}
tools={"inspect":{"scope":"read"},"repair":{"scope":"write","max_rows":50},"drop":{"scope":"admin"}}
calls=[{"tool":"inspect","scope":"read"},{"tool":"repair","scope":"write","rows":20},{"tool":"repair","scope":"write","rows":80},{"tool":"drop","scope":"write"}]
print(calls)

## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S26 — DATA-1284: Agentic Data Operations** → [`../work_simulations/26_acme_commerce_agentic_data_operations/README.md`](../work_simulations/26_acme_commerce_agentic_data_operations/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
def authorize(call):
    spec=tools.get(call["tool"])
    if not spec: return False,"unknown"
    if call["scope"]!=spec["scope"] and not (spec["scope"]=="read" and call["scope"] in {"read","write"}): return False,"scope"
    if call["tool"]=="repair" and call.get("rows",0)>spec["max_rows"]: return False,"row_budget"
    if call["tool"]=="drop": return False,"admin_required"
    return True,"ok"
for c in calls: print(c,authorize(c))

In [ ]:
# Break: stale state makes the repair target unsafe.
state["table_version"]=8
planned_version=7
print("stale plan",planned_version!=state["table_version"])

In [ ]:
# Improve: require a fresh version immediately before a write.
def execute(call,expected_version):
    if state["table_version"]!=expected_version: return "refresh_required"
    ok,reason=authorize(call); return "executed" if ok else reason
print(execute(calls[1],7)); print(execute(calls[1],8))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production agentic operations need least privilege, tool isolation, audit traces, approval boundaries, idempotency, kill switches and explicit rollback paths.

## 9. Independent challenge
Add a retrying agent that repeats a write after a timeout. Design the idempotency key and maximum action budget before execution.

### Evidence to keep
authorization results, stale-plan detection, fresh-state gate and action policy

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

## Agent-native engineering challenge

### The new professional boundary
An implementation agent can produce a plausible solution for **agentic data operations and tool authority**. Your job is to decide whether that solution is correct and safe.

**Agent can do:** generate an implementation, tests or a first diagnosis.

**You must understand:** the mechanism, assumptions and failure boundary.

**You must verify:** choose evidence that could disprove the agent's answer.

**You must decide:** ship, revise, reject or escalate.

### Challenge
Ask an agent (or use the deterministic exercise harness in this notebook) to propose a solution. Do not accept the output because it runs. Record one plausible failure mode, create an adversarial case that exposes it, and capture the evidence.


In [ ]:
# Deterministic agent-review harness: intentionally small and vendor-neutral.
agent_proposal = {
    "task": 'agentic data operations and tool authority',
    "status": "proposed",
    "assumptions": ["happy-path input", "single failure boundary"],
    "needs_human_verification": True,
}
review = {
    "understood_mechanism": True,
    "verification_designed": True,
    "adversarial_case": "required",
    "decision": "revise-until-evidence-supports-release",
}
print(agent_proposal)
print(review)
